# 03 - 模型评测 (三层评测体系)

在微调前后分别运行完整的三层评测体系，量化模型能力提升。

- **Layer 1**: 通用能力基准 (MMLU-Pro, GPQA, HumanEval, MATH, BBH)
- **Layer 2**: TRIZ定制评测 (原理识别、矛盾解决、案例质量、ARIZ完整性)
- **Layer 3**: 工程性能基准 (吞吐量、延迟、内存)

**注意**: 为避免OOM，模型以 **4-bit量化** 加载，lm-eval复用同一模型实例。

In [ ]:
import sys
sys.path.append('/home/meerkat/mongoose_ai')

import os
import torch
from datetime import datetime
from transformers import BitsAndBytesConfig
from utils.training_utils import load_model_and_tokenizer
from utils.pipeline_state import PipelineState
from config import BASE_MODEL, MODELS_DIR, RESULTS_DIR, BENCHMARK_CONFIG

# 确定模型路径
model_path = os.path.join(MODELS_DIR, BASE_MODEL.split('/')[-1])

print(f"评测模型: {model_path}")
print("使用4-bit量化加载模型 (节省内存，避免OOM)...")

quantization_config = {
    "load_in_4bit": True,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_compute_dtype": "float16",
    "bnb_4bit_use_double_quant": True,
}

model, tokenizer = load_model_and_tokenizer(
    model_name_or_path=model_path,
    quantization_config=quantization_config,
    device_map='auto',
    trust_remote_code=True,
)

print("\n模型加载完成!")
print(f"模型类型: {model.config.model_type if hasattr(model.config, 'model_type') else 'unknown'}")
print(f"显存占用: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

In [ ]:
# 注册基准评测开始 — BENCH-05
state = PipelineState()

state.register(
    name="baseline_run",
    path=str(RESULTS_DIR),
    artifact_type="benchmark",
    metadata={
        "model_path": model_path,
        "model_dtype": "4-bit_nf4",
        "timestamp": datetime.now().isoformat(),
        "status": "running",
    }
)

print("基准评测已注册到 pipeline_state")

## 3.2 Layer 1: 通用能力基准 (可选)

**注意**: 运行完整的lm-eval评测需要较长时间(数小时)，建议在微调前后各运行一次以对比。

In [ ]:
# Layer 1: 通用能力评测 — BENCH-01, BENCH-06
from utils.benchmark_utils import run_lm_evaluation

# 从配置读取评测任务
general_tasks_config = BENCHMARK_CONFIG['general_benchmarks']
tasks = list(general_tasks_config.keys())

print(f"开始通用能力评测 (Layer 1): {tasks}")
print("注意: 完整评测需要数小时，如时间有限可只运行部分任务")
print("(BENCH-06 可选: 如时间紧张可跳过此单元格)")

# 运行评测 (复用已加载的4-bit模型，避免重复加载导致OOM)
general_results = run_lm_evaluation(
    model_path=model_path,
    tasks=tasks,
    output_dir=RESULTS_DIR,
    num_fewshot=5,
    batch_size=1,
    model=model,
    tokenizer=tokenizer,
)

print("\nLayer 1 评测完成!")

## 3.3 Layer 2: TRIZ定制评测

In [ ]:
# Layer 2: TRIZ领域自定义评测 — BENCH-03 (用户要求: 基线也运行Layer 2)
from utils.benchmark_utils import run_triz_evaluation
from config import DATA_DIR

print("开始TRIZ领域评测 (Layer 2)...")
print("注意: 在基线阶段运行Layer 2可建立完整的before/after对比")

# 从sample_data.json加载测试数据 (使用test split或全部样本)
test_data_path = os.path.join(DATA_DIR, "sample_data.json")

triz_results = run_triz_evaluation(
    model=model,
    tokenizer=tokenizer,
    test_data_path=test_data_path,
    output_dir=RESULTS_DIR,
    max_new_tokens=512,
    temperature=0.7,
)

print("\nLayer 2 评测完成!")
print(f"TRIZ评测结果: {triz_results}")


## 3.4 Layer 3: 工程性能基准

In [ ]:
# Layer 3: 性能评测
from utils.benchmark_utils import run_performance_benchmark

print("开始性能评测 (Layer 3)...")

perf_results = run_performance_benchmark(
    model=model,
    tokenizer=tokenizer,
    output_dir=RESULTS_DIR,
    max_tokens=512,
)

print("\n性能评测完成!")

## 3.5 生成综合评测报告

In [ ]:
# 聚合评测结果并持久化 — BENCH-05
from utils.benchmark_utils import aggregate_results
from pathlib import Path

report = aggregate_results(
    general_results=general_results if 'general_results' in dir() else None,
    triz_results=triz_results if 'triz_results' in dir() else None,
    perf_results=perf_results if 'perf_results' in dir() else None,
    output_dir=RESULTS_DIR,
)

print("\n综合评测报告生成完成!")

# 注册基准结果到 pipeline_state
result_files = list(Path(RESULTS_DIR).glob('*_results_*.json'))
latest_result = max(result_files, key=lambda p: p.stat().st_mtime) if result_files else None

# 记录原始 Layer 1 lm-eval JSON 路径和紧凑摘要 — BENCH-05
layer1_files = list(Path(RESULTS_DIR).glob('lm_eval_results_*.json'))
latest_layer1_file = max(layer1_files, key=lambda p: p.stat().st_mtime) if layer1_files else None

layer1_path = str(latest_layer1_file) if latest_layer1_file else None
layer1_summary = {}
if 'general_results' in dir() and general_results and 'tasks' in dir() and tasks:
    for task in tasks:
        task_results = general_results.get("results", {}).get(task, {})
        for candidate in ["acc_norm", "acc", "exact_match", "pass_at_1"]:
            if candidate in task_results:
                layer1_summary[task] = {candidate: task_results[candidate]}
                break
        else:
            for key, value in task_results.items():
                if isinstance(value, (int, float)) and ("acc" in key or "score" in key):
                    layer1_summary[task] = {key: value}
                    break
# 提取关键指标用于元数据
triz_summary = {}
if 'triz_results' in dir() and triz_results:
    triz_summary = {
        "principle_accuracy": triz_results.get("principle_accuracy", "N/A"),
        "contradiction_resolution": triz_results.get("contradiction_resolution", "N/A"),
        "case_quality_bleu": triz_results.get("case_quality_bleu", "N/A"),
        "ariz_completeness": triz_results.get("ariz_completeness", "N/A"),
    }

state.register(
    name="baseline_results",
    path=str(latest_result) if latest_result else str(RESULTS_DIR),
    artifact_type="benchmark",
    metadata={
        "model_path": model_path,
        "model_dtype": "4-bit_nf4",
        "tasks": tasks if "tasks" in dir() else [],
        "layer2_included": True,
        "triz_summary": triz_summary,
        "perf_throughput": perf_results.get("throughput_tokens_per_sec", "N/A") if "perf_results" in dir() else "N/A",
        "layer1_path": layer1_path,
        "layer1_summary": layer1_summary,
        "timestamp": datetime.now().isoformat(),
    }
)

# 更新 baseline_run 状态为已完成
state.register(
    name="baseline_run",
    path=str(RESULTS_DIR),
    artifact_type="benchmark",
    metadata={
        "model_path": model_path,
        "model_dtype": "4-bit_nf4",
        "timestamp": datetime.now().isoformat(),
        "status": "completed",
        "layers_run": ["Layer 1 (general)", "Layer 2 (TRIZ)", "Layer 3 (performance)"],
    }
)

print(f"基准结果已持久化到 pipeline_state")
print(f"报告保存位置: {RESULTS_DIR}")

# 显示摘要
summary = state.summary()
print(f"\nPipeline状态摘要: {summary}")

## 3.6 清理显存

In [ ]:
# 清理显存，为训练做准备
del model
del tokenizer
torch.cuda.empty_cache()

print("显存已清理")
print(f"当前显存占用: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

---

## 下一步

评测完成！记录基线分数后，请打开: **04_qlora_finetune.ipynb** 进行模型微调